# Chroma CRUD Operations

This notebook walks through create, read, update, and delete operations with a local Chroma vector store.

In [1]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEndpointEmbeddings

## 1. Set Up Paths and the Vector Store

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('e:/Projects/Campusx-Advance-Rag/Code/Advanced_Rag_Codes/04_vector_stores')

In [3]:
# Load environment variables from the local .env file.
# dotenv_path = project_root / ".env"
# load_dotenv(dotenv_path=dotenv_path)

# if not os.getenv("OPENAI_API_KEY"):
#     raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

# print(f"Loaded environment from: {dotenv_path}")

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "Hugging Face token not found. Add HF_TOKEN=... to your .env file."
    )

MODEL = "BAAI/bge-small-en-v1.5"
PROVIDER = "hf-inference"

print(f"Model: {MODEL}")
print(f"Provider: {PROVIDER}")

Model: BAAI/bge-small-en-v1.5
Provider: hf-inference


In [4]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo_2"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: e:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\04_vector_stores\notebooks\chroma_langchain_db


In [5]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

# Make sure the parent directory exists.
persist_directory.parent.mkdir(parents=True, exist_ok=True)

Removed the old Chroma directory.


In [6]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-small-en-v1.5",
    huggingfacehub_api_token=HF_TOKEN
)

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


## 2. Add Small Helper Functions

In [7]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [8]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [9]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [10]:
print(uuid4())

4d98e4f5-8a80-42f1-b355-c6035ed6637f


In [11]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=5bfd1506-a3e6-4257-880e-a80df21242ff
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=de8a9d34-bea7-4b70-9a20-b5d97c1f2aaa
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=16c71654-689c-41dc-877d-935b27a73cab
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=36e168da-eaaf-4b66-911b-7989bed5834f
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=c842ffc1-2664-499a-afa8-649c81d404dd
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=88a93ee2-df1c-471b-ba68-a53f3f84bd72
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [12]:
documents[0].id

'5bfd1506-a3e6-4257-880e-a80df21242ff'

In [13]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
5bfd1506-a3e6-4257-880e-a80df21242ff
de8a9d34-bea7-4b70-9a20-b5d97c1f2aaa
16c71654-689c-41dc-877d-935b27a73cab
36e168da-eaaf-4b66-911b-7989bed5834f
c842ffc1-2664-499a-afa8-649c81d404dd
88a93ee2-df1c-471b-ba68-a53f3f84bd72
91c606a2-4c72-4768-aa18-b2b0bc0e20d2
28b38d32-fdef-4196-af09-a0e555c34f5f
8459c936-ee13-4195-86dd-b89cacbfc7ab
ac5b631e-5ac7-4d7a-86e4-1a7aa6531975

Total inserted documents: 10


## 4. Read the Stored Data Back

In [14]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [15]:
raw_records

{'ids': ['5bfd1506-a3e6-4257-880e-a80df21242ff',
  'de8a9d34-bea7-4b70-9a20-b5d97c1f2aaa',
  '16c71654-689c-41dc-877d-935b27a73cab',
  '36e168da-eaaf-4b66-911b-7989bed5834f',
  'c842ffc1-2664-499a-afa8-649c81d404dd',
  '88a93ee2-df1c-471b-ba68-a53f3f84bd72',
  '91c606a2-4c72-4768-aa18-b2b0bc0e20d2',
  '28b38d32-fdef-4196-af09-a0e555c34f5f',
  '8459c936-ee13-4195-86dd-b89cacbfc7ab',
  'ac5b631e-5ac7-4d7a-86e4-1a7aa6531975'],
 'embeddings': array([[-0.04186527,  0.0022468 ,  0.03420469, ...,  0.02753688,
          0.0569935 , -0.05768351],
        [-0.03659178, -0.00234231,  0.01409155, ...,  0.0402372 ,
         -0.01763388, -0.08176838],
        [-0.00871275, -0.04957389, -0.00743502, ..., -0.00705291,
          0.04954195, -0.05502832],
        ...,
        [-0.11348923,  0.02073365,  0.02570359, ...,  0.1192267 ,
          0.00336366,  0.00961999],
        [-0.01797233,  0.05248439, -0.03686758, ...,  0.00863248,
          0.02897388, -0.01043756],
        [-0.04017704,  0.05733875, 

In [16]:
print(raw_records["embeddings"][0:2, 0:20].shape)

(2, 20)


In [17]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
5bfd1506-a3e6-4257-880e-a80df21242ff
de8a9d34-bea7-4b70-9a20-b5d97c1f2aaa
16c71654-689c-41dc-877d-935b27a73cab


In [18]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['28b38d32-fdef-4196-af09-a0e555c34f5f',
 '8459c936-ee13-4195-86dd-b89cacbfc7ab',
 'ac5b631e-5ac7-4d7a-86e4-1a7aa6531975']

In [19]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=28b38d32-fdef-4196-af09-a0e555c34f5f
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=8459c936-ee13-4195-86dd-b89cacbfc7ab
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=ac5b631e-5ac7-4d7a-86e4-1a7aa6531975
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



In [20]:
print(selected_documents)

[Document(id='28b38d32-fdef-4196-af09-a0e555c34f5f', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'), Document(id='8459c936-ee13-4195-86dd-b89cacbfc7ab', metadata={'topic': 'Cricket', 'doc_number': 9}, page_content='Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.'), Document(id='ac5b631e-5ac7-4d7a-86e4-1a7aa6531975', metadata={'doc_number': 10, 'topic': 'Cricket'}, page_content='A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.')]


## 5. Run a Similarity Search

In [21]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [22]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=36e168da-eaaf-4b66-911b-7989bed5834f
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=28b38d32-fdef-4196-af09-a0e555c34f5f
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=c842ffc1-2664-499a-afa8-649c81d404dd
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [23]:
search_results

[Document(id='36e168da-eaaf-4b66-911b-7989bed5834f', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='28b38d32-fdef-4196-af09-a0e555c34f5f', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='c842ffc1-2664-499a-afa8-649c81d404dd', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [24]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='36e168da-eaaf-4b66-911b-7989bed5834f', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.4810202717781067),
 (Document(id='28b38d32-fdef-4196-af09-a0e555c34f5f', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.5335822105407715),
 (Document(id='c842ffc1-2664-499a-afa8-649c81d404dd', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  0.6080049276351929),
 (Document(id='91c606a2-4c72-4768-aa18-b2b0bc0e20d2', metadata={'topic': 'LLM', 'doc_number': 7}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  0.6502256393432617)]

## 6. Update Existing Documents

In [25]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['36e168da-eaaf-4b66-911b-7989bed5834f',
 '28b38d32-fdef-4196-af09-a0e555c34f5f']

In [26]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=36e168da-eaaf-4b66-911b-7989bed5834f
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=28b38d32-fdef-4196-af09-a0e555c34f5f
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [27]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [28]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
36e168da-eaaf-4b66-911b-7989bed5834f
28b38d32-fdef-4196-af09-a0e555c34f5f


In [29]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=36e168da-eaaf-4b66-911b-7989bed5834f
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=28b38d32-fdef-4196-af09-a0e555c34f5f
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [30]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [31]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=36e168da-eaaf-4b66-911b-7989bed5834f
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=c842ffc1-2664-499a-afa8-649c81d404dd
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [32]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['8459c936-ee13-4195-86dd-b89cacbfc7ab',
 'ac5b631e-5ac7-4d7a-86e4-1a7aa6531975']

In [33]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
8459c936-ee13-4195-86dd-b89cacbfc7ab
ac5b631e-5ac7-4d7a-86e4-1a7aa6531975


In [34]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
5bfd1506-a3e6-4257-880e-a80df21242ff
de8a9d34-bea7-4b70-9a20-b5d97c1f2aaa
16c71654-689c-41dc-877d-935b27a73cab
36e168da-eaaf-4b66-911b-7989bed5834f
c842ffc1-2664-499a-afa8-649c81d404dd
88a93ee2-df1c-471b-ba68-a53f3f84bd72
91c606a2-4c72-4768-aa18-b2b0bc0e20d2
28b38d32-fdef-4196-af09-a0e555c34f5f

Deleted ids still present?
8459c936-ee13-4195-86dd-b89cacbfc7ab: False
ac5b631e-5ac7-4d7a-86e4-1a7aa6531975: False


In [35]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
